In [1]:
# =====================================================================
# SIMPLE, HIGH-QUALITY EDA + PREPROCESSING
# Bangla + English + Banglish SMS (NORMAL / PROMO / SPAM)
# Final CSV contains ONLY 7 useful columns.
# =====================================================================

from pathlib import Path
import hashlib
import html
import json
import os
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")

# =====================================================================
# 1. PATHS
# =====================================================================
POSSIBLE_INPUTS = [
    Path("/kaggle/input/datasets/olidali/unified-bangla-sms-dataset/unified_bangla_sms_dataset.csv"),
    Path("/kaggle/input/unified-bangla-sms-dataset/unified_bangla_sms_dataset.csv"),
    Path("/home/user/uploads/unified_bangla_sms_dataset.csv"),  # local fallback
]
INPUT_PATH = next((p for p in POSSIBLE_INPUTS if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Change INPUT_PATH in the code. Checked:\n" +
        "\n".join(map(str, POSSIBLE_INPUTS))
    )

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/simple_bangla_sms_output")
else:
    OUTPUT_DIR = Path(os.environ.get(
        "OUTPUT_DIR", "/home/user/deliverables/simple_bangla_sms_output"
    ))
PLOT_DIR = OUTPUT_DIR / "eda_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Input :", INPUT_PATH)
print("Output:", OUTPUT_DIR)

# =====================================================================
# 2. LOAD AND VALIDATE
# =====================================================================
raw = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=True, encoding="utf-8")
required_columns = {"text", "label", "source_dataset"}
missing_columns = required_columns - set(raw.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

raw = raw[["text", "label", "source_dataset"]].copy()
raw.insert(0, "original_row", np.arange(len(raw), dtype=int))
raw["label"] = raw["label"].astype("string").str.strip().str.upper()
raw["source_dataset"] = raw["source_dataset"].astype("string").str.strip()

valid_labels = {"NORMAL", "PROMO", "SPAM"}
unknown_labels = set(raw["label"].dropna().unique()) - valid_labels
if unknown_labels:
    raise ValueError(f"Unexpected labels: {sorted(unknown_labels)}")

print("\nOriginal shape:", raw.shape)
print("\nOriginal labels:\n", raw["label"].value_counts())
print("\nOriginal sources:\n", raw["source_dataset"].value_counts())

# =====================================================================
# 3. SAFE MULTILINGUAL TEXT CLEANING
# =====================================================================
# This cleaning keeps Bangla, English, Banglish, other Unicode languages,
# URLs, numbers, punctuation and emoji. It does NOT apply stopword removal,
# stemming or transliteration.

INVISIBLE_RE = re.compile(
    r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f"
    r"\u200b\u200c\u200d\u2060\u061c\u200e\u200f"
    r"\u202a-\u202e\u2066-\u2069\ufeff]"
)
# Explicit tags preserve Android OTP marker <#>.
HTML_TAG_RE = re.compile(
    r"</?(?:a|b|br|div|em|font|html|body|i|li|p|span|strong|table|td|th|tr|ul|ol)"
    r"(?:\s+[^<>]{0,300})?/?>",
    flags=re.IGNORECASE,
)
SPACE_RE = re.compile(r"\s+")
CHAR_MAP = str.maketrans({
    "“": '"', "”": '"', "„": '"',
    "‘": "'", "’": "'", "`": "'",
    "–": "-", "—": "-", "−": "-",
    "…": "...",
})

def clean_text(value):
    """Conservative, language-preserving Unicode text cleaning."""
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFKC", text)
    text = HTML_TAG_RE.sub(" ", text)
    text = INVISIBLE_RE.sub("", text)
    text = text.translate(CHAR_MAP)
    text = SPACE_RE.sub(" ", text).strip()
    return text

raw["text_clean"] = raw["text"].map(clean_text)
raw["clean_key"] = raw["text_clean"].str.casefold()

# =====================================================================
# 4. REMOVE EMPTY, CONFLICTING AND EXACT-DUPLICATE ROWS
# =====================================================================
# Policy:
# - empty cleaned text: remove;
# - exactly the same cleaned text with different labels: remove the full group;
# - exactly the same cleaned text with the same label: retain one copy.

empty_mask = raw["text_clean"].eq("")
nonempty = raw.loc[~empty_mask].copy()

labels_per_text = nonempty.groupby("clean_key")["label"].nunique()
conflicting_keys = set(labels_per_text[labels_per_text > 1].index)
conflict_mask = nonempty["clean_key"].isin(conflicting_keys)
conflict_rows = nonempty.loc[conflict_mask].copy()

cleaned = nonempty.loc[~conflict_mask].copy()
cleaned = cleaned.sort_values("original_row")
exact_duplicate_rows = int(cleaned.duplicated("clean_key", keep="first").sum())
cleaned = cleaned.drop_duplicates("clean_key", keep="first").copy()
rows_after_exact_cleaning = len(cleaned)

print("\nEmpty rows removed:", int(empty_mask.sum()))
print("Conflicting-label groups removed:", len(conflicting_keys))
print("Rows in conflicting groups removed:", len(conflict_rows))
print("Same-label exact duplicate rows removed:", exact_duplicate_rows)
print("Rows after exact cleaning:", rows_after_exact_cleaning)

# =====================================================================
# 5. CREATE SIMPLE MASKED TEXT
# =====================================================================
# text_clean is best for transformers.
# text_masked is useful for TF-IDF/classical ML because changing phone numbers,
# amounts and URLs do not create a completely new sentence pattern.

EMAIL_RE = re.compile(r"(?i)\b[A-Z0-9._%+\-]+@[A-Z0-9.\-]+\.[A-Z]{2,}\b")
URL_RE = re.compile(
    r"(?i)(?:(?:https?://|www\.)[^\s<>\"']+|"
    r"\b(?:[a-z0-9](?:[a-z0-9\-]{0,62}[a-z0-9])?\.)+"
    r"(?:com|org|net|edu|gov|mil|io|co|info|biz|me|ly|app|xyz|top|site|online|bd)"
    r"(?:/[^\s<>\"']*)?)"
)
USSD_RE = re.compile(r"\*[0-9০-৯*]+#")
PHONE_RE = re.compile(
    r"(?<![0-9০-৯])(?:\+?(?:88|৮৮))?(?:0|০)(?:1|১)[3-9৩-৯]"
    r"(?:[\s.\-]?[0-9০-৯]){8}(?![0-9০-৯])"
)
CURRENCY_RE = re.compile(r"(?i)(?:[৳$€£₹]|\b(?:tk|taka|bdt|usd|inr)\b)")
NUMBER_RE = re.compile(r"[0-9০-৯]+(?:[.,][0-9০-৯]+)?")
REPEATED_PUNCT_RE = re.compile(r"([!?.,।#*\-_=+])\1{2,}")
SHORT_URL_RE = re.compile(
    r"(?i)\b(?:bit\.ly|cutt\.ly|tinyurl\.com|t\.co|goo\.gl|is\.gd|rb\.gy|"
    r"rebrand\.ly|shorturl\.at|tiny\.cc)(?:/|\b)"
)

def mask_text(text):
    x = EMAIL_RE.sub(" EMAILTOKEN ", text)
    x = URL_RE.sub(" URLTOKEN ", x)
    x = USSD_RE.sub(" USSDTOKEN ", x)
    x = PHONE_RE.sub(" PHONETOKEN ", x)
    x = CURRENCY_RE.sub(" MONEYTOKEN ", x)
    x = NUMBER_RE.sub(" NUMTOKEN ", x)
    x = REPEATED_PUNCT_RE.sub(r"\1\1", x)
    return SPACE_RE.sub(" ", x).strip().casefold()

cleaned["text_masked"] = cleaned["text_clean"].map(mask_text)

# =====================================================================
# 6. SIMPLE SCRIPT TYPE
# =====================================================================
def script_type(text):
    has_bengali = any("\u0980" <= ch <= "\u09FF" for ch in text)
    has_latin = any("a" <= ch.casefold() <= "z" for ch in text)
    if has_bengali and has_latin:
        return "BENGALI_LATIN_MIXED"
    if has_bengali:
        return "BENGALI_ONLY"
    if has_latin:
        return "LATIN_ONLY"
    return "OTHER_SCRIPT"

cleaned["script_type"] = cleaned["text_clean"].map(script_type)

# =====================================================================
# 7. CONTROL NEAR-DUPLICATE TEMPLATE REPETITION
# =====================================================================
# Internally build a template key from text_masked. One representative is kept
# for each template + label. This prevents thousands of automatically generated
# number/date variations from dominating both training and evaluation.
# The template ID is used internally and is NOT exported to the simple CSV.

def template_key(masked_text):
    # Keep letters, combining marks and numbers from any Unicode language.
    chars = []
    for ch in masked_text:
        category = unicodedata.category(ch)
        chars.append(ch if category[0] in {"L", "M", "N"} or ch == "_" else " ")
    return SPACE_RE.sub(" ", "".join(chars)).strip().casefold()

cleaned["template_key"] = cleaned["text_masked"].map(template_key)
cleaned["template_group"] = cleaned["template_key"].map(
    lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()[:16]
)

before_template_cleaning = len(cleaned)
cleaned = cleaned.sort_values("original_row")
cleaned = cleaned.drop_duplicates(
    subset=["template_group", "label"], keep="first"
).copy()
template_variants_removed = before_template_cleaning - len(cleaned)

print("Near-duplicate same-label template variants removed:", template_variants_removed)
print("Final rows:", len(cleaned))

# =====================================================================
# 8. LEAKAGE-SAFE 80/10/10 SPLIT
# =====================================================================
# All retained rows belonging to the same template stay in one split.
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
cleaned = cleaned.reset_index(drop=True)
cleaned["fold"] = -1

for fold, (_, validation_index) in enumerate(
    sgkf.split(cleaned, y=cleaned["label"], groups=cleaned["template_group"])
):
    cleaned.loc[validation_index, "fold"] = fold

cleaned["split"] = np.select(
    [cleaned["fold"].eq(0), cleaned["fold"].eq(1)],
    ["test", "validation"],
    default="train",
)

# Quality checks: no template may cross splits.
assert cleaned.groupby("template_group")["split"].nunique().max() == 1
assert cleaned["text_clean"].ne("").all()
assert not cleaned.duplicated(["template_group", "label"]).any()

LABEL_TO_ID = {"NORMAL": 0, "PROMO": 1, "SPAM": 2}
cleaned["label_id"] = cleaned["label"].map(LABEL_TO_ID).astype("int8")

# =====================================================================
# 9. EXPORT THE SIMPLE FINAL DATASET: EXACTLY 7 COLUMNS
# =====================================================================
final = cleaned.rename(columns={"text": "text_original"})[
    [
        "text_original",   # original SMS
        "text_clean",      # transformer/general NLP input
        "text_masked",     # TF-IDF/classical ML input
        "label",           # original label
        "label_id",        # NORMAL=0, PROMO=1, SPAM=2
        "script_type",     # Bengali / Latin / mixed
        "split",           # train / validation / test
    ]
].copy()

assert final.shape[1] == 7
assert final["text_clean"].notna().all()
assert set(final["label"]) == valid_labels
assert set(final["split"]) == {"train", "validation", "test"}

FINAL_PATH = OUTPUT_DIR / "unified_bangla_sms_preprocessed_7_columns.csv"
final.to_csv(FINAL_PATH, index=False, encoding="utf-8-sig")

# =====================================================================
# 10. EDA IMAGES
# =====================================================================
LABEL_ORDER = ["NORMAL", "PROMO", "SPAM"]
COLORS = {"NORMAL": "#2A9D8F", "PROMO": "#E9C46A", "SPAM": "#E76F51"}

# Image 1: final class distribution
fig, ax = plt.subplots(figsize=(8, 5))
counts = final["label"].value_counts().reindex(LABEL_ORDER)
sns.barplot(x=counts.index, y=counts.values, palette=COLORS, ax=ax)
for i, value in enumerate(counts.values):
    ax.text(i, value + counts.max() * 0.015,
            f"{value:,}\n({value/len(final):.1%})", ha="center", fontsize=10)
ax.set_title("Final Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of SMS")
fig.tight_layout()
fig.savefig(PLOT_DIR / "01_class_distribution.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Image 2: original source-label distribution
source_percent = pd.crosstab(
    raw["source_dataset"], raw["label"], normalize="index"
).reindex(columns=LABEL_ORDER) * 100
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    source_percent, annot=True, fmt=".1f", cmap="YlOrRd",
    cbar_kws={"label": "% within source"}, ax=ax
)
ax.set_title("Original Label Distribution by Source")
ax.set_xlabel("Class")
ax.set_ylabel("Source dataset")
fig.tight_layout()
fig.savefig(PLOT_DIR / "02_source_label_heatmap.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Image 3: language-script profile by class
script_table = pd.crosstab(final["script_type"], final["label"]).reindex(
    columns=LABEL_ORDER, fill_value=0
)
fig, ax = plt.subplots(figsize=(10, 5.5))
script_table.plot(
    kind="bar", stacked=True,
    color=[COLORS[x] for x in LABEL_ORDER], ax=ax
)
ax.set_title("Script Type by Class")
ax.set_xlabel("Script type")
ax.set_ylabel("Number of SMS")
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Class")
fig.tight_layout()
fig.savefig(PLOT_DIR / "03_script_type_by_class.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Image 4: message length distribution
plot_data = final.copy()
plot_data["character_length"] = plot_data["text_clean"].str.len()
p99 = plot_data["character_length"].quantile(0.99)
plot_data["length_for_plot"] = plot_data["character_length"].clip(upper=p99)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(
    data=plot_data, x="length_for_plot", hue="label",
    hue_order=LABEL_ORDER, palette=COLORS, bins=45,
    element="step", stat="density", common_norm=False, ax=axes[0]
)
axes[0].set_title(f"Message Length Distribution (display clipped at p99={p99:.0f})")
axes[0].set_xlabel("Characters")
sns.boxplot(
    data=plot_data, x="label", y="length_for_plot",
    order=LABEL_ORDER, palette=COLORS, showfliers=False, ax=axes[1]
)
axes[1].set_title("Message Length by Class")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Characters")
fig.tight_layout()
fig.savefig(PLOT_DIR / "04_message_length.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Image 5: important SMS structure patterns
eda_features = pd.DataFrame({
    "label": final["label"],
    "Has URL": final["text_clean"].str.contains(URL_RE, regex=True, na=False),
    "Has phone": final["text_clean"].str.contains(PHONE_RE, regex=True, na=False),
    "Has USSD": final["text_clean"].str.contains(USSD_RE, regex=True, na=False),
    "Has currency": final["text_clean"].str.contains(CURRENCY_RE, regex=True, na=False),
    "Has short URL": final["text_clean"].str.contains(SHORT_URL_RE, regex=True, na=False),
})
pattern_percent = eda_features.groupby("label").mean(numeric_only=True).mul(100).reindex(LABEL_ORDER)
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.heatmap(
    pattern_percent, annot=True, fmt=".1f", cmap="Blues",
    cbar_kws={"label": "% of class"}, ax=ax
)
ax.set_title("Important SMS Patterns by Class")
ax.set_xlabel("Pattern")
ax.set_ylabel("Class")
fig.tight_layout()
fig.savefig(PLOT_DIR / "05_sms_patterns.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# Image 6: row-cleaning flow
flow_labels = [
    "Original",
    "After empty + conflicts",
    "After exact duplicates",
    "Final after templates",
]
flow_values = [
    len(raw),
    len(raw) - int(empty_mask.sum()) - len(conflict_rows),
    rows_after_exact_cleaning,
    len(final),
]
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=flow_values, y=flow_labels, color="#457B9D", ax=ax)
for i, value in enumerate(flow_values):
    ax.text(value + max(flow_values) * 0.01, i, f"{value:,}", va="center")
ax.set_title("Preprocessing Row Flow")
ax.set_xlabel("Number of rows")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(PLOT_DIR / "06_preprocessing_flow.png", dpi=220, bbox_inches="tight")
plt.close(fig)

# =====================================================================
# 11. SAVE SIMPLE QUALITY SUMMARY
# =====================================================================
summary = {
    "original_rows": int(len(raw)),
    "empty_rows_removed": int(empty_mask.sum()),
    "exact_conflicting_label_groups_removed": int(len(conflicting_keys)),
    "rows_in_conflicting_groups_removed": int(len(conflict_rows)),
    "same_label_exact_duplicate_rows_removed": int(exact_duplicate_rows),
    "rows_after_exact_cleaning": int(rows_after_exact_cleaning),
    "same_label_template_variants_removed": int(template_variants_removed),
    "final_rows": int(len(final)),
    "final_columns": int(final.shape[1]),
    "label_counts": {k: int(v) for k, v in final["label"].value_counts().items()},
    "split_counts": {k: int(v) for k, v in final["split"].value_counts().items()},
    "label_id_mapping": LABEL_TO_ID,
    "random_seed": SEED,
}
with open(OUTPUT_DIR / "quality_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Save excluded exact-conflict records separately for research audit.
conflict_rows[["original_row", "text", "text_clean", "label", "source_dataset"]].to_csv(
    OUTPUT_DIR / "conflicting_label_rows_for_review.csv",
    index=False, encoding="utf-8-sig"
)

# Final reload test
check = pd.read_csv(FINAL_PATH, encoding="utf-8-sig")
assert check.shape == final.shape
assert check.columns.tolist() == [
    "text_original", "text_clean", "text_masked", "label",
    "label_id", "script_type", "split"
]
assert check["text_clean"].notna().all()

print("\n" + "=" * 68)
print("DONE: SIMPLE HIGH-QUALITY DATASET")
print("=" * 68)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nFinal columns:", final.columns.tolist())
print("\nSplit by label:\n", pd.crosstab(final["split"], final["label"], margins=True))
print("\nFinal CSV:", FINAL_PATH)
print("EDA images:", PLOT_DIR)

Input : /kaggle/input/datasets/olidali/unified-bangla-sms-dataset/unified_bangla_sms_dataset.csv
Output: /kaggle/working/simple_bangla_sms_output

Original shape: (24582, 4)

Original labels:
 label
NORMAL    9470
SPAM      8785
PROMO     6327
Name: count, dtype: Int64

Original sources:
 source_dataset
BTTC                       10283
Bengali_SMS_Smishing        7005
Labeled_Bangla_SMS_2026     3999
BangalaBarta                2772
Financial_Scams              523
Name: count, dtype: Int64

Empty rows removed: 1
Conflicting-label groups removed: 309
Rows in conflicting groups removed: 814
Same-label exact duplicate rows removed: 2798
Rows after exact cleaning: 20969
Near-duplicate same-label template variants removed: 4429
Final rows: 16540

DONE: SIMPLE HIGH-QUALITY DATASET
{
  "original_rows": 24582,
  "empty_rows_removed": 1,
  "exact_conflicting_label_groups_removed": 309,
  "rows_in_conflicting_groups_removed": 814,
  "same_label_exact_duplicate_rows_removed": 2798,
  "rows_after